# MSigDB ORA Saturation: study subsampling, 1% only (seed1)

**Environment:** `clamp-analyses`

Single rs_pct slice of `00_bp_saturation_study_ora_analysis.ipynb`,
scoped to **1% study coverage**, sweeping all available K within
this level, **seed1 only** (preliminary results). Same method, same
cache paths (`rs{rs_pct}_k{{k}}_seed1_msigdb.rds`) as the all-in-one
notebook, so `01_bp_saturation_study_plot.ipynb` reads this with no
changes.

`pvalueCutoff = 0.05` (not `1`) — see the all-in-one notebook for why:
with no filtering, `enricher()` returns every one of 35k MSigDB terms
per LV (with a verbose `geneID` column), which OOM-killed a K=1728 run
at ~80GB RSS. `0.05` is the loosest FDR threshold used downstream and is
behavior-preserving for the coverage calculation.

In [ ]:
library(here)
library(clusterProfiler)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/08_saturation_study")
output_dir <- here("output/03_model_biology/00_archs4/08_saturation_study/00_bp_saturation_study_ora_analysis")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"), recursive = TRUE, showWarnings = FALSE)

rs_pct <- 1L
seed   <- 1L  # seed1 only (preliminary)

## Load MSigDB gene sets

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
message(sprintf("MSigDB gene sets loaded: %d", length(unique(msig_gmt$term))))

## Model grid: K values available for rs_pct=1%, seed=1

Discovered dynamically from disk (filtered to this rs_pct and seed only).

In [ ]:
all_subdirs <- list.dirs(models_dir, recursive = FALSE, full.names = FALSE)

pattern <- sprintf("^study_saturation_rs%d_k([0-9]+)_seed_%d$", rs_pct, seed)

model_grid <- do.call(rbind, lapply(all_subdirs, function(d) {
  m <- regmatches(d, regexec(pattern, d))[[1]]
  if (length(m) < 2) return(NULL)
  z_path_full <- file.path(models_dir, d, "CLAMPfull_hall", "Z.csv")
  if (!file.exists(z_path_full)) return(NULL)
  data.frame(
    k_val       = as.integer(m[2]),
    subdir      = d,
    z_path_full = z_path_full,
    z_path_base = file.path(models_dir, d, "CLAMPbase", "Z.csv"),
    stringsAsFactors = FALSE
  )
}))

if (is.null(model_grid) || nrow(model_grid) == 0) {
  stop("No models found for rs_pct=1, seed=1 in: ", models_dir)
}

model_grid <- model_grid[order(model_grid$k_val), ]
rownames(model_grid) <- NULL

message("Available K values: ", paste(model_grid$k_val, collapse = ", "))

## Helper: get n_studies from subsample_info

`subsample_info.rds` lives in the upstream `07_bp_coverage_study` source dirs.

In [ ]:
study_base_dir <- here("output/01_model_building/04_archs4/07_bp_coverage_study")
study_dirs     <- list.dirs(study_base_dir, recursive = FALSE, full.names = FALSE)

pct_to_subdir <- list()
for (d in study_dirs) {
  m <- regmatches(d, regexec("([0-9]+)$", d))[[1]]
  if (length(m) >= 2) pct_to_subdir[[as.character(as.integer(m[2]))]] <- d
}

get_n_studies <- function(rs_pct, seed_idx) {
  subdir <- pct_to_subdir[[as.character(rs_pct)]]
  if (is.null(subdir)) return(NA_integer_)
  src_path <- file.path(
    study_base_dir, subdir,
    sprintf("study_coverage_rs%d_seed_%d", rs_pct, seed_idx),
    "subsample_info.rds"
  )
  if (file.exists(src_path)) readRDS(src_path)$n_studies else NA_integer_
}

## Helper: run ORA for one model

Returns a list with raw `terms_padj` (minimum p.adjust per MSigDB term across all LVs).

In [ ]:
run_ora_for_model <- function(z_path) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)

  term_overlap   <- tapply(msig_gmt$gene %in% universe_genes, msig_gmt$term, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })

  # n_samples via B.csv header only (avoids loading the full multi-GB rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  ora_results <- lapply(seq_len(n_lvs), function(i) {
    genes <- top_genes_per_lv[, i]
    tryCatch(
      clusterProfiler::enricher(
        gene          = genes,
        universe      = universe_genes,
        TERM2GENE     = msig_gmt,
        pAdjustMethod = "BH",
        pvalueCutoff  = 0.05,
        qvalueCutoff  = 1,
        minGSSize     = 10,
        maxGSSize     = 50000
      ),
      error = function(e) NULL
    )
  })

  all_dfs <- Filter(Negate(is.null), lapply(ora_results, function(r) {
    if (is.null(r) || nrow(as.data.frame(r)) == 0) return(NULL)
    as.data.frame(r)
  }))

  if (length(all_dfs) == 0) {
    warning("No ORA results returned for: ", z_path)
    return(NULL)
  }

  combined <- do.call(rbind, all_dfs)

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(combined$p.adjust, combined$ID, min)
  )
}

## Run ORA: CLAMPfull

In [ ]:
for (i in seq_len(nrow(model_grid))) {
  row <- model_grid[i, ]
  z_path <- row$z_path_full

  cache_path <- file.path(
    output_dir, "CLAMPfull",
    sprintf("rs%d_k%d_seed%d_msigdb.rds", rs_pct, row$k_val, seed)
  )
  label <- sprintf("CLAMPfull rs%d%% k%d seed%d", rs_pct, row$k_val, seed)

  if (!file.exists(z_path)) {
    message(sprintf("[SKIP]    %s (no Z.csv)", label))
  } else if (file.exists(cache_path)) {
    message(sprintf("[CACHED]  %s", label))
  } else {
    t0 <- proc.time()[[3]]
    message(sprintf("[START]   %s", label))
    res <- run_ora_for_model(z_path)
    elapsed <- round(proc.time()[[3]] - t0)
    if (!is.null(res)) {
      res$n_studies <- get_n_studies(rs_pct, seed)
      saveRDS(res, cache_path)
      message(sprintf("[DONE]    %s in %ds -> saved", label, elapsed))
    } else {
      message(sprintf("[FAILED]  %s after %ds", label, elapsed))
    }
  }
}

## Run ORA: CLAMPbase

In [ ]:
for (i in seq_len(nrow(model_grid))) {
  row <- model_grid[i, ]
  z_path <- row$z_path_base

  cache_path <- file.path(
    output_dir, "CLAMPbase",
    sprintf("rs%d_k%d_seed%d_msigdb.rds", rs_pct, row$k_val, seed)
  )
  label <- sprintf("CLAMPbase rs%d%% k%d seed%d", rs_pct, row$k_val, seed)

  if (!file.exists(z_path)) {
    message(sprintf("[SKIP]    %s (no Z.csv)", label))
  } else if (file.exists(cache_path)) {
    message(sprintf("[CACHED]  %s", label))
  } else {
    t0 <- proc.time()[[3]]
    message(sprintf("[START]   %s", label))
    res <- run_ora_for_model(z_path)
    elapsed <- round(proc.time()[[3]] - t0)
    if (!is.null(res)) {
      res$n_studies <- get_n_studies(rs_pct, seed)
      saveRDS(res, cache_path)
      message(sprintf("[DONE]    %s in %ds -> saved", label, elapsed))
    } else {
      message(sprintf("[FAILED]  %s after %ds", label, elapsed))
    }
  }
}